(sec-1)=
# 1. Definición del problema y dataset

En esta sección se formaliza el problema en notación matemática, se declaran por anticipado la métrica, el margen de relevancia y la unidad de independencia, y se justifica y describe técnicamente el conjunto de datos. Esto es primordial porque estas decisiones condicionan todo lo que sigue y se heredan en los Entregables 2 y 3.

(sec-1-1)=
## 1.1 Definición formal del problema

El sistema recibe los atributos de un flujo de red dentro de un enjambre de drones y debe indicar si es tráfico normal o a qué ataque pertenece. Formalmente:

- **Espacio de entrada:** $\mathcal{X} = \mathbb{R}^{d}$, el vector de atributos de tráfico y desempeño de un flujo, con $d$ fijado en el análisis exploratorio.
- **Espacio de salida:** $\mathcal{Y}$ = {Blackhole, Flooding, Normal, Sybil, Wormhole}.
- **Función objetivo:** $f : \mathcal{X} \to \mathcal{Y}$, aproximada por $f_\theta(x) = \arg\max_{c \in \mathcal{Y}} p_\theta(c \mid x)$.
- **Función de pérdida:** entropía cruzada, $\mathcal{L}(\theta) = -\frac{1}{n}\sum_{i=1}^{n} w_{y_i} \log p_\theta(y_i \mid x_i)$, con los pesos $w_c$ definidos en el análisis exploratorio.

**Tipo de problema.** El proyecto corresponde al tipo 1 del enunciado, clasificación con datos tabulares, en su versión multiclase. Se opta por las cinco categorías y no por la versión binaria de normal frente a ataque, porque reconocer el tipo de ataque responde una pregunta más exigente y de ella se puede derivar la binaria, mientras que lo contrario no es posible.

**Pregunta de investigación.** ¿Qué nivel de desempeño, medido con F1 macro, establecen los tres modelos de Deep Learning más básicos de la literatura al clasificar los flujos de un enjambre de drones en tráfico normal y cuatro ataques, cuando la partición respeta la estructura de dependencia del archivo, y qué mejora debe superar un modelo avanzado para considerarse relevante?

**Hipótesis operativa.** Al menos un par de modelos base difiere en F1 macro en una magnitud igual o mayor que el margen de relevancia práctica δ.

**Declaración a priori.** Las decisiones siguientes se fijan antes de ver resultados de cualquier modelo de Deep Learning, para evitar la exploración selectiva. La métrica primaria es el F1 macro, $F1_{macro} = \frac{1}{K}\sum_{c=1}^{K} \frac{2 P_c R_c}{P_c + R_c}$, que promedia sin ponderar las $K = 5$ clases, porque en seguridad un ataque poco frecuente importa tanto como uno frecuente. El margen δ = 0,01 tiene una lectura directa: por la definición del promedio macro, equivale a que el F1 de una sola clase cambie 0,05 mientras las demás permanecen iguales, y se contrasta con la variabilidad medida en el piloto de la sección {ref}`4.3.1 <sec-4-3-1>`. La unidad de independencia es la serie, una de las que propone el enunciado, porque los flujos consecutivos según `FlowID` son muy parecidos entre sí, de modo que la dependencia sigue a la posición en el archivo y no a una entidad como el cliente, como se demuestra en la sección 2.1.3. Como el archivo no identifica la simulación de origen, cada serie se aproxima con bloques de 1000 flujos contiguos. Esto no convierte el proyecto en uno de series de tiempo: el archivo no trae marca de tiempo, de modo que no se afirma que ese orden sea cronológico, y ningún modelo usa rezagos como entrada, así que sigue siendo clasificación tabular. Estas decisiones y la partición se fijaron el 3 de noviembre de 2026, antes de entrenar. Al día siguiente solo se cambió el nombre de la unidad para usar el término del enunciado; la partición y las corridas son las mismas.

In [3]:
DECISIONES = {
    "fecha": "2026-11-03",
    "tipo_problema": "1. Clasificación con datos tabulares, multiclase de cinco categorías",
    "metrica_primaria": "F1 macro en el conjunto de prueba",
    "metricas_secundarias": ["accuracy", "balanced accuracy", "precision, recall y F1 (macro, ponderado y por clase)", "AUC-ROC macro", "AUC-PR macro", "MCC", "ECE"],
    "margen_delta": 0.01,
    "alfa": 0.05,
    "unidad_independencia": "serie: flujos consecutivos según FlowID, aproximada con bloques de 1000 flujos contiguos",
    "particion": {"entrenamiento": 0.64, "validacion": 0.16, "prueba": 0.20, "semilla": SEMILLA, "estrategia": "estratificada por clase y agrupada por bloques"},
}
(RESULTADOS / "declaracion_a_priori.json").write_text(json.dumps(DECISIONES, ensure_ascii=False, indent=1), encoding="utf-8")
display(pd.DataFrame({"Decisión": ["Fecha", "Métrica primaria", "Métricas secundarias", "Margen δ", "Nivel α", "Unidad de independencia", "Partición"],
                      "Valor": ["3 de noviembre de 2026", DECISIONES["metrica_primaria"], ", ".join(DECISIONES["metricas_secundarias"]), DECISIONES["margen_delta"], DECISIONES["alfa"],
                                DECISIONES["unidad_independencia"], "64 / 16 / 20, estratificada por clase y agrupada por bloques, semilla 42"]}).style.hide(axis="index"))

Decisión,Valor
Fecha,3 de noviembre de 2026
Métrica primaria,F1 macro en el conjunto de prueba
Métricas secundarias,"accuracy, balanced accuracy, precision, recall y F1 (macro, ponderado y por clase), AUC-ROC macro, AUC-PR macro, MCC, ECE"
Margen δ,"0,0100"
Nivel α,"0,0500"
Unidad de independencia,"serie: flujos consecutivos según FlowID, aproximada con bloques de 1000 flujos contiguos"
Partición,"64 / 16 / 20, estratificada por clase y agrupada por bloques, semilla 42"


(sec-1-2)=
## 1.2 Justificación del dataset

Los enjambres de drones se comunican por enlaces inalámbricos y enrutan el tráfico entre ellos mismos, sin una infraestructura fija, de modo que un nodo malicioso puede interrumpir la misión desde adentro. Detectar esa intrusión a partir del tráfico es primordial porque no exige modificar el protocolo de los drones. Los cuatro ataques del conjunto de datos representan esa amenaza: en Blackhole, un nodo atrae rutas y descarta los paquetes; en Flooding, satura la red; en Sybil, se presenta con varias identidades, y en Wormhole, dos nodos crean un túnel que altera las rutas.

El conjunto es público, en Kaggle, Zenodo (licencia CC BY 4.0) e IEEE DataPort (DOI 10.21227/j5p4-zt27). Según sus autores, los flujos se generaron por simulación en NS-3.24, con movilidad de enjambre basada en un algoritmo BOID extendido, IEEE 802.11ac, enrutamiento AODV, tráfico UDP y canal Nakagami, con un alcance de transmisión de 100 m y 20 dBm de potencia (Zeng et al., 2025). Es decir, son datos simulados y no capturas de drones reales, lo que limita su validez externa.

Hay referencias directas para comparar el baseline. Los autores reportan, con partición aleatoria estratificada de 80 y 20 %, un F1 de 82 % con regresión logística, 93 % con DNN, 93 % con CNN, 94 % con LSTM y 96 % con XGBoost (Zeng et al., 2025). Zarkadis y Douligeris (2026) reportan un F1 de prueba de 90,66 % con MLP y de 94,28 % con RealMLP, y Bakiri et al. (2026), con una partición por nodo de origen, una exactitud de 92,50 % en cinco clases.

(sec-1-3)=
## 1.3 Descripción técnica

Cada fila es un flujo de red, con 22 atributos y la etiqueta. Los autores los agrupan así:

| Grupo | Atributos | Uso en los modelos |
|---|---|---|
| Conexión | `FlowID`, `SrcAddr`, `SrcPort`, `DstAddr`, `DstPort`, `Protocol` | Se excluyen, por las razones que se miden en la sección {ref}`2.1.4 <sec-2-1-4>` |
| Volumen de tráfico | `FlowDuration/s`, `TxPackets`, `RxPackets`, `LostPackets`, `TxBytes`, `RxBytes`, `TxPacketRate/s`, `RxPacketRate/s`, `TxByteRate/s`, `RxByteRate/s`, `MeanPacketSize` | Entrada |
| Desempeño | `MeanDelay/s`, `MeanJitter/s`, `Throughput/Kbps`, `PacketDropRate`, `AverageHopCount` | Entrada |

La etiqueta no proviene de anotadores: el simulador anota cada flujo con la etiqueta de su escenario, normal o de ataque, y una verificación posterior cruza los tiempos de inyección del ataque con los flujos capturados (Zeng et al., 2025). Por tal motivo, puede haber flujos sin relación con el ataque que reciben la etiqueta de su escenario, un ruido de etiqueta que se examina en la sección {ref}`2.1.2 <sec-2-1-2>`. En cuanto a la estructura de dependencia, los flujos de una misma simulación comparten nodos, tiempos y condiciones, y el archivo no identifica esa simulación, de modo que la dependencia se audita de forma indirecta en la sección {ref}`2.1.3 <sec-2-1-3>`.

In [4]:
tipos = datos.dtypes.astype(str).value_counts().to_dict()
display(pd.DataFrame({"Propiedad": ["Filas", "Columnas", "Tamaño del archivo (MB)", "Tipos de columna", "Versión", "SHA-256"],
                      "Valor": [N, datos.shape[1], round(CSV.stat().st_size / 1e6, 1), str(tipos), "Kaggle y Zenodo, v1", sha]}).style.hide(axis="index"))
interpretar(f'''**Interpretación.** El archivo tiene {num(N, 0)} flujos y {datos.shape[1]} columnas, y pesa {num(CSV.stat().st_size / 1e6, 1)} MB, de modo que cabe completo en la memoria y se puede entrenar en CPU. La suma SHA-256 de la tabla identifica esta versión, y el cuaderno se detiene si el archivo cambia.

Esto significa que el tamaño no impone ninguna restricción sobre el diseño, y que las repeticiones del estudio se pueden definir por criterios estadísticos y no por el costo de cómputo.

Por lo anterior, se trabaja con el archivo completo, sin muestreo, salvo en los diagnósticos donde se indique.''')

Propiedad,Valor
Filas,122171
Columnas,23
Tamaño del archivo (MB),"20,1000"
Tipos de columna,"{'float64': 10, 'int64': 9, 'str': 4}"
Versión,"Kaggle y Zenodo, v1"
SHA-256,d50d339f68be7b23f0bf089dd438b20a1835c13182d8641220538121440164d0


**Interpretación.** El archivo tiene 122 171 flujos y 23 columnas, y pesa 20,1 MB, de modo que cabe completo en la memoria y se puede entrenar en CPU. La suma SHA-256 de la tabla identifica esta versión, y el cuaderno se detiene si el archivo cambia.

Esto significa que el tamaño no impone ninguna restricción sobre el diseño, y que las repeticiones del estudio se pueden definir por criterios estadísticos y no por el costo de cómputo.

Por lo anterior, se trabaja con el archivo completo, sin muestreo, salvo en los diagnósticos donde se indique.